In [ ]:
!pip install -q groq tenacity

In [ ]:
import os
import getpass
from groq import Groq

# Colab Secrets are optional; this also works in a local Jupyter environment.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

api_key = None
if userdata is not None:
    try:
        api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Groq API key: ").strip()

if not api_key:
    raise ValueError("GROQ_API_KEY is required. Add it to Colab Secrets or enter it when prompted.")

os.environ["GROQ_API_KEY"] = api_key
client = Groq(api_key=api_key)

# Current Groq production model; change this value to switch models.
MODEL_NAME = "openai/gpt-oss-120b"
print(f"Groq client initialized. Model: {MODEL_NAME}")

Groq client initialized. Model: openai/gpt-oss-120b


### 🛠️ Define the Groq Model Caller
Use Groq's Chat Completions API. The model ID is configurable through `MODEL_NAME` above.

In [ ]:
def call_model(prompt: str) -> str:
    """Send a prompt to Groq and return the generated text.

    API exceptions are allowed to propagate so the retry decorator below
    can retry rate-limit errors instead of accidentally swallowing them.
    """
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_completion_tokens=500
    )

    if completion.choices and completion.choices[0].message.content:
        return completion.choices[0].message.content.strip()

    return "No response generated."

In [ ]:
import re
import json
import time
import groq

In [ ]:
print("Groq model caller ready ✅")

Groq model caller ready ✅


### 🛠️ Handle Groq API Rate Limits (HTTP 429)
Groq applies model-specific request and token limits. This retry wrapper retries rate-limit errors with exponential backoff. The actual limits depend on your Groq plan and model.

In [ ]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
    retry_if_exception_type
)

@retry(
    wait=wait_random_exponential(min=1, max=10),
    stop=stop_after_attempt(3),
    retry=retry_if_exception_type(groq.RateLimitError),
    reraise=True
)
def call_model_with_retry(prompt: str) -> str:
    return call_model(prompt)

The `financial_risk_explainer` function below uses `call_model_with_retry()` so temporary Groq rate-limit errors can be retried automatically.

In [ ]:
INVESTMENT_ADVICE_PATTERNS = [
    r"should i invest",
    r"which mutual fund is best",
    r"recommend.{0,20}(stock|mutual fund|fund)",
    r"tell me.{0,20}(stock|fund) to buy",
    r"where should i put my money",
    r"best stock to buy",
    r"guaranteed returns",
    r"suggest.{0,20}investment",
    r"if you were me.{0,40}(invest|investment)",
    r"what would you personally choose.{0,30}(invest|investment)",
    r"is it a good time to invest",
    r"where would you invest"
]

def is_investment_advice_request(user_input: str) -> bool:
    lower_input = user_input.lower()
    return any(re.search(pattern, lower_input) for pattern in INVESTMENT_ADVICE_PATTERNS)

In [ ]:
FINANCE_EXPLAINER_PROMPT = """
You are a financial education assistant.

ROLE:
Explain financial concepts clearly in an educational way.

STRICT RULES:
1. NEVER recommend specific stocks or mutual funds.
2. NEVER provide personalized investment advice.
3. NEVER tell the user what they should invest in.
4. Only provide general educational information.
5. Explain risks and uncertainty where relevant.

If the user asks for specific investment advice, politely refuse and offer
general educational information instead.

User question: {user_input}
"""

In [ ]:
def filter_investment_advice(response: str):
    risky_phrases = [
        r"you should invest",
        r"i recommend",
        r"best mutual fund is",
        r"buy this stock",
        r"guaranteed returns",
        r"you should buy"
    ]

    for pattern in risky_phrases:
        if re.search(pattern, response.lower()):
            return (
                "I cannot provide personalized investment advice. "
                "This tool is for educational purposes only.",
                True
            )

    return response, False

In [ ]:
def financial_risk_explainer(user_input: str):
    if not isinstance(user_input, str) or not user_input.strip():
        return {
            "input": user_input,
            "blocked": False,
            "block_reason": "Empty or invalid input",
            "response": "Please enter a financial question.",
            "filtered": False,
            "risk_classification": "invalid"
        }

    user_input = user_input.strip()

    result = {
        "input": user_input,
        "blocked": False,
        "block_reason": None,
        "response": None,
        "filtered": False,
        "risk_classification": "safe"
    }

    # Pre-call guardrail: block requests for personalized investment advice.
    if is_investment_advice_request(user_input):
        result["blocked"] = True
        result["block_reason"] = "Pre-call: Investment advice request detected"
        result["risk_classification"] = "high_risk"
        result["response"] = (
            "I cannot provide personalized investment advice.\\n\\n"
            "This tool is designed only for educational explanations "
            "about financial concepts such as risk and diversification.\\n\\n"
            "Disclaimer: This information is for educational purposes only."
        )
        return result

    prompt = FINANCE_EXPLAINER_PROMPT.format(user_input=user_input)
    ai_response = call_model_with_retry(prompt)

    # Post-call filter: prevent advice-like model output from being shown.
    filtered_response, was_filtered = filter_investment_advice(ai_response)

    disclaimer = (
        "\\n\\n---\\n"
        "Disclaimer: This information is for educational purposes only "
        "and should not be considered financial advice."
    )

    result["response"] = filtered_response + disclaimer
    result["filtered"] = was_filtered
    return result

In [ ]:
test_queries = [
    "What is diversification?",
    "Explain market risk in mutual funds",
    "Should I invest in HDFC mutual fund?",
    "Tell me the best stock to buy now"
]

for query in test_queries:
    print(f"\nUser: {query}")
    result = financial_risk_explainer(query)
    print("Blocked:", result["blocked"])
    print("Response:\n", result["response"])
    print("\n" + "="*60)
    time.sleep(1)


User: What is diversification?
Blocked: False
Response:
 **Diversification** is a risk‑management strategy that involves spreading your investments across a variety of assets, sectors, geographic regions, or other categories. The core idea is that a portfolio made up of many different kinds of investments is less likely to suffer a large loss from any single adverse event.

### How Diversification Works

| Dimension of diversification | Example | Why it helps |
|------------------------------|---------|--------------|
| **Asset class** | Stocks, bonds, real estate, cash, commodities | Different asset classes often react differently to economic conditions. When stocks are down, bonds may hold up or even rise. |
| **Industry / sector** | Technology, healthcare, consumer staples, utilities | A downturn in one industry (e.g., a tech slump) may not affect another (e.g., utilities), reducing overall volatility. |
| **Geography** | U.S., Europe, emerging markets, Asia‑Pacific | Economic cycl

In [ ]:
safe_queries = [
    "What is a mutual fund?",
    "Explain diversification in simple terms",
    "What are the types of investment risks?"
]

blocked_queries = [
    "Should I invest in HDFC mutual fund?",
    "Which mutual fund is best right now?",
    "Where should I put my money for high returns?"
]

tricky_queries = [
    "If you were me, where would you invest?",
    "Is it a good time to invest in gold?",
    "What would you personally choose to invest in?"
]

all_test_queries = safe_queries + blocked_queries + tricky_queries

for query in all_test_queries:
    print(f"\nUser: {query}")
    result = financial_risk_explainer(query)
    print("Blocked:", result["blocked"])
    print("Risk Classification:", result["risk_classification"])
    print("Response:\n", result["response"])
    print("\n" + "="*60)
    time.sleep(1)


User: What is a mutual fund?
Blocked: False
Risk Classification: safe
Response:
 ### What Is a Mutual Fund?

A **mutual fund** is an investment vehicle that pools money from many individual investors to buy a diversified portfolio of securities—such as stocks, bonds, money‑market instruments, or a mix of these. The fund is managed by professional portfolio managers who make the day‑to‑day investment decisions on behalf of the shareholders.

| Feature | Description |
|---------|-------------|
| **Pooling of Capital** | Each investor buys **shares** (or units) of the fund. The total assets of the fund equal the sum of all investors’ contributions. |
| **Professional Management** | A fund manager (or team) selects the underlying securities according to the fund’s stated investment objective and strategy. |
| **Diversification** | Because the fund holds many securities, an individual investor gains exposure to a broad set of assets with a single purchase, reducing the impact of any one se

In [ ]:
print(call_model("What is diversification in simple terms"))

**Diversification** means “don’t put all your eggs in one basket.”  

In simple terms, it’s the practice of spreading your resources—like money, time, or effort—across a variety of different things so that if one of them does poorly, the others can help keep everything balanced.

### How it works in everyday life
- **Investing:** Instead of buying only one company’s stock, you buy many stocks, bonds, real‑estate, etc. If one company’s share price drops, the others may stay steady or rise, reducing the overall loss.
- **Career:** Learning several skills (e.g., coding, writing, public speaking) makes you less vulnerable if one job market shrinks.
- **Food:** Eating a mix of fruits, vegetables, proteins, and grains gives you a broader range of nutrients than eating only one type of food.

### Why it helps
1. **Reduces risk:** Bad performance in one area is offset by good performance elsewhere.
2. **Smooths out results:** Your overall outcome tends to be more stable over time.
3. **Opens o

In [ ]:
!pip install -q gradio groq tenacity

In [ ]:
import gradio as gr

def gradio_interface(user_input):
    try:
        result = financial_risk_explainer(user_input)
        return result["response"]
    except groq.RateLimitError:
        return "Groq rate limit reached. Please wait a little and try again."
    except Exception as e:
        return f"System Error: {str(e)}"

interface = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(lines=2, placeholder="Ask about mutual funds, risk, diversification..."),
    outputs=gr.Textbox(label="Educational explanation"),
    title="Financial Risk Explanation Tool",
    description=(
        "Educational AI that explains financial concepts but does not provide "
        "personalized investment advice. Do not enter sensitive personal or financial information."
    )
)

# Keep the app private by default. Set share=True only if you intentionally
# want to create a publicly accessible Gradio link.
interface.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
test_prompt = "Explain what diversification means in investing."
print(call_model(test_prompt))

**Diversification** is a risk‑management strategy that spreads your investment dollars across a variety of assets, sectors, geographies, and/or investment styles so that the performance of any single holding has a limited impact on the overall portfolio.

---

## 1. Why Diversify?

| Goal | How Diversification Helps |
|------|----------------------------|
| **Reduce volatility** | Losses in one area can be offset by gains (or smaller losses) elsewhere. |
| **Protect against “unknown unknowns”** | If a particular industry or region suffers an unexpected shock, the rest of the portfolio may be insulated. |
| **Improve risk‑adjusted returns** | A well‑diversified mix can achieve a similar expected return with lower overall risk (measured by standard deviation, Sharpe ratio, etc.). |
| **Avoid concentration risk** | Prevents over‑reliance on a single stock, bond, or asset class that could dominate the portfolio’s outcome. |

---

## 2. The Core Idea: Correlation

- **Correlation** measures